In [1]:
import threading

counter = 0

def increment_unsafe():
    global counter
    for _ in range(100_000):
        counter += 1   # NOT thread-safe


counter = 0

t1 = threading.Thread(target=increment_unsafe)
t2 = threading.Thread(target=increment_unsafe)

t1.start()
t2.start()

t1.join()
t2.join()

print("Unsafe result:", counter, "(expected 200000)")

Unsafe result: 200000 (expected 200000)


In [2]:
import threading

counter = 0
lock = threading.Lock()

def increment_safe():
    global counter
    for _ in range(100_000):
        with lock:
            counter += 1


counter = 0

t1 = threading.Thread(target=increment_safe)
t2 = threading.Thread(target=increment_safe)

t1.start()
t2.start()

t1.join()
t2.join()

print("Safe result:", counter, "(expected 200000)")

Safe result: 200000 (expected 200000)


In [3]:
import time
import threading

counter = 0

def increment_unsafe():
    global counter
    for _ in range(100_000):
        counter += 1

start = time.perf_counter()

t1 = threading.Thread(target=increment_unsafe)
t2 = threading.Thread(target=increment_unsafe)

t1.start(); t2.start()
t1.join(); t2.join()

end = time.perf_counter()

print("Unsafe result:", counter)
print("Unsafe time:", end - start)

counter = 0
lock = threading.Lock()

def increment_safe():
    global counter
    for _ in range(100_000):
        with lock:
            counter += 1

start = time.perf_counter()

t1 = threading.Thread(target=increment_safe)
t2 = threading.Thread(target=increment_safe)

t1.start(); t2.start()
t1.join(); t2.join()

end = time.perf_counter()

print("Safe result:", counter)
print("Safe time:", end - start)

Unsafe result: 200000
Unsafe time: 0.04943220000131987
Safe result: 200000
Safe time: 0.14864239998860285


In [4]:
import threading
import queue
import time
import random

BUFFER_SIZE = 5
buffer = queue.Queue(maxsize=BUFFER_SIZE)

def producer(n_items):
    for i in range(n_items):
        item = random.randint(1,100)
        buffer.put(item)
        print("Produced:", item, "| Buffer size:", buffer.qsize())
        time.sleep(random.uniform(0.1,0.3))

def consumer(n_items):
    for _ in range(n_items):
        item = buffer.get()
        print("Consumed:", item, "| Buffer size:", buffer.qsize())
        time.sleep(random.uniform(0.2,0.5))


N = 10

p = threading.Thread(target=producer, args=(N,))
c = threading.Thread(target=consumer, args=(N,))

p.start()
c.start()

p.join()
c.join()

print("Producer-Consumer finished")

Produced: 39 | Buffer size: 1
Consumed: 39 | Buffer size: 0
Produced: 92 | Buffer size: 1
Consumed: 92 | Buffer size: 0
Produced: 80 | Buffer size: 1
Consumed: 80 | Buffer size: 0
Produced: 49 | Buffer size: 1
Produced: 2 | Buffer size: 2
Consumed: 49 | Buffer size: 1
Produced: 87 | Buffer size: 2
Consumed: 2 | Buffer size: 1
Produced: 85 | Buffer size: 2
Produced: 77 | Buffer size: 3
Consumed: 87 | Buffer size: 2
Produced: 79 | Buffer size: 3
Consumed: 85 | Buffer size: 2
Produced: 12 | Buffer size: 3
Consumed: 77 | Buffer size: 2
Consumed: 79 | Buffer size: 1
Consumed: 12 | Buffer size: 0
Producer-Consumer finished


In [5]:
import threading
import queue
import time
import random

#Change buffer size to 2 
BUFFER_SIZE = 2
buffer = queue.Queue(maxsize=BUFFER_SIZE)

def producer(n_items):
    for i in range(n_items):
        item = random.randint(1,100)
        buffer.put(item)
        print("Produced:", item, "| Buffer size:", buffer.qsize())
        time.sleep(random.uniform(0.1,0.3))

def consumer(n_items):
    for _ in range(n_items):
        item = buffer.get()
        print("Consumed:", item, "| Buffer size:", buffer.qsize())
        time.sleep(random.uniform(0.2,0.5))


N = 10

p = threading.Thread(target=producer, args=(N,))
c = threading.Thread(target=consumer, args=(N,))

p.start()
c.start()

p.join()
c.join()

print("Producer-Consumer finished")

Produced: 53 | Buffer size: 1
Consumed: 53 | Buffer size: 0
Produced: 28 | Buffer size: 1
Consumed: 28 | Buffer size: 0
Produced: 86 | Buffer size: 1
Produced: 53 | Buffer size: 2
Consumed: 86 | Buffer size: 1
Produced: 91 | Buffer size: 2
Consumed:Produced: 16 | Buffer size: 2
 53 | Buffer size: 1
Consumed:Produced: 14 | Buffer size: 2
 91 | Buffer size: 1
Consumed:Produced: 74 | Buffer size: 2
 16 | Buffer size: 1
Consumed:Produced: 41 | Buffer size: 2
 14 | Buffer size: 1
Consumed:Produced: 99 | Buffer size: 2
 74 | Buffer size: 1
Consumed: 41 | Buffer size: 1
Consumed: 99 | Buffer size: 0
Producer-Consumer finished


In [7]:
import threading
import time

forks = [threading.Lock() for _ in range(5)]

def philosopher(i):
    left = forks[i]
    right = forks[(i+1)%5]

    while True:
        print(f"Philosopher {i} thinking")
        time.sleep(1)

        print(f"Philosopher {i} hungry")

        left.acquire()
        right.acquire()

        print(f"Philosopher {i} eating")
        time.sleep(1)

        left.release()
        right.release()


for i in range(5):
    threading.Thread(target=philosopher, args=(i,), daemon=True).start()

time.sleep(10)


def philosopher_safe(i):
    first = min(i,(i+1)%5)
    second = max(i,(i+1)%5)

    while True:
        print(f"Philosopher {i} thinking")
        time.sleep(1)

        forks[first].acquire()
        forks[second].acquire()

        print(f"Philosopher {i} eating")
        time.sleep(1)

        forks[first].release()
        forks[second].release()

Philosopher 0 thinking
Philosopher 1 thinking
Philosopher 2 thinking
Philosopher 3 thinking
Philosopher 4 thinking
Philosopher 0 hungry
Philosopher 0 eating
Philosopher 1 hungry
Philosopher 2 hungry
Philosopher 2 eating
Philosopher 3 hungry
Philosopher 4 hungry
Philosopher 0 thinkingPhilosopher 4 eating

Philosopher 2 thinking
Philosopher 1 eating
Philosopher 4 thinkingPhilosopher 2 hungry
Philosopher 1 thinking

Philosopher 3 eating
Philosopher 0 hungry
Philosopher 0 eating
Philosopher 1 hungryPhilosopher 0 thinking
Philosopher 3 thinking
Philosopher 2 eating
Philosopher 4 hungry
Philosopher 4 eating

Philosopher 0 hungry
Philosopher 4 thinking
Philosopher 2 thinking
Philosopher 1 eating
Philosopher 3 hungry
Philosopher 3 eating
Philosopher 4 hungry
Philosopher 1 thinking
Philosopher 3 thinking
Philosopher 2 hungry
Philosopher 2 eating
Philosopher 0 eating
Philosopher 1 hungry
Philosopher 0 thinking
Philosopher 3 hungry
Philosopher 4 eating
Philosopher 2 thinking
Philosopher 1 eating


In [6]:
import threading
import time

event = threading.Event()

def sensor():
    time.sleep(3)
    print("Sensor triggered!")
    event.set()

def monitor():
    print("Waiting for sensor...")
    event.wait()
    print("ALERT: Sensor event detected!")


threading.Thread(target=monitor).start()
threading.Thread(target=sensor).start()

Waiting for sensor...
Sensor triggered!
ALERT: Sensor event detected!
